# Data Preperation and Imports

In [1]:
import pandas as pd
import numpy as np
import sys 
sys.path.append('../')
import tools
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from scipy import stats as scipy_stats  # Rename to avoid conflict
import vectorbt as vbt


In [2]:

raw_data = pd.read_csv('../datasets/SPY_1SECOND_2020-10-18_to_2025-10-19.csv', 
                 parse_dates=['Timestamp'])
                 #nrows=10000000)  # Start with 100k rows for testing



rth_end = pd.to_datetime("16:00:00").time()
rth_start = pd.to_datetime("09:30:00").time()

mask = raw_data['Timestamp'].dt.time.between(rth_start, rth_end)
raw_data = raw_data.loc[mask].reset_index(drop=True)


print("Data loaded successfully:")
print(f"- Time range: {raw_data['Timestamp'].min()} to {raw_data['Timestamp'].max()}")
print(f"- Number of rows: {len(raw_data)}")
print("\nSample data:")
print(raw_data.head())

Data loaded successfully:
- Time range: 2020-10-19 09:32:15+00:00 to 2025-10-17 13:52:52+00:00
- Number of rows: 6141258

Sample data:
                  Timestamp    Open    High     Low   Close  Volume
0 2020-10-19 09:32:15+00:00  349.16  349.16  349.16  349.16     170
1 2020-10-19 09:34:55+00:00  349.06  349.06  349.06  349.06     500
2 2020-10-19 09:35:25+00:00  349.13  349.13  349.13  349.13     500
3 2020-10-19 09:35:32+00:00  349.11  349.11  349.09  349.10    2500
4 2020-10-19 09:35:34+00:00  349.11  349.11  349.11  349.11     300


In [3]:
# Set the time span to display (edit this variable)
display_range = (raw_data['Timestamp'].iloc[0], raw_data['Timestamp'].iloc[10000])  #  first 10000 points

mask = (raw_data['Timestamp'] >= display_range[0]) & (raw_data['Timestamp'] <= display_range[1])


fig = go.Figure()
fig.add_trace(go.Scatter(x=raw_data.loc[mask, 'Timestamp'], 
                        y=raw_data.loc[mask, 'Close'], 
                        mode='lines', 
                        name='Price'))


fig.show()



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [4]:
tools = tools.TechnicalTools(raw_data)
data = tools.compute_features(price_col="Close", time_col="Timestamp", target_col=1)

# Trade on Probability from the Last 5 Bars

### Concept
Given a sequence of recent price movements, we want to estimate what tends to happen next.

Let:
- **G** = Green bar (price increased)  
- **R** = Red bar (price decreased)

At any given moment, the **last 5 bars** can be categorized into one of several patterns depending on how many red bars appear.

---

###  All 5-Bar Combinations

#### **0 Red Bars (1 Combination)**
1. (G, G, G, G, G)

---

#### **1 Red Bar (5 Combinations)**
2. (G, G, G, G, R)  
3. (G, G, G, R, G)  
4. (G, G, R, G, G)  
5. (G, R, G, G, G)  
6. (R, G, G, G, G)

---

#### **2 Red Bars (10 Combinations)**
7. (G, G, G, R, R)  
8. (G, G, R, G, R)  
9. (G, G, R, R, G)  
10. (G, R, G, G, R)  
11. (G, R, G, R, G)  
12. (G, R, R, G, G)  
13. (R, G, G, G, R)  
14. (R, G, G, R, G)  
15. (R, G, R, G, G)  
16. (R, R, G, G, G)

---

#### **3 Red Bars (10 Combinations)**
17. (G, G, R, R, R)  
18. (G, R, G, R, R)  
19. (G, R, R, G, R)  
20. (G, R, R, R, G)  
21. (R, G, G, R, R)  
22. (R, G, R, G, R)  
23. (R, G, R, R, G)  
24. (R, R, G, G, R)  
25. (R, R, G, R, G)  
26. (R, R, R, G, G)

---

#### **4 Red Bars (5 Combinations)**
27. (G, R, R, R, R)  
28. (R, G, R, R, R)  
29. (R, R, G, R, R)  
30. (R, R, R, G, R)  
31. (R, R, R, R, G)

---

#### **5 Red Bars (1 Combination)**
32. (R, R, R, R, R)

---

###  Mathematical Insight

For any sequence length **n = 5**, and number of red bars **k**,  
the number of unique combinations is given by the **binomial coefficient**:

\[
\text{# of combinations} = \binom{n}{k} = \frac{n!}{k!(n-k)!}
\]

Hence:

| # of R | Combinations | Formula |
|:------:|:-------------:|:--------:|
| 0 | 1 | \( \binom{5}{0} \) |
| 1 | 5 | \( \binom{5}{1} \) |
| 2 | 10 | \( \binom{5}{2} \) |
| 3 | 10 | \( \binom{5}{3} \) |
| 4 | 5 | \( \binom{5}{4} \) |
| 5 | 1 | \( \binom{5}{5} \) |
| **Total** | **32** | \( 2^5 \) |

---

### Application

For each of these 32 combinations:
1. **Look ahead** to the next bar(s).  
2. Compute:
   - The **percentage change** in price, or  
   - Whether the next move was **up (G)** or **down (R)**.  
3. **Aggregate statistics**:
   - Average future return per pattern, or  
   - Probability of next bar being green vs. red.

---

### Trading Logic

Use these historical probabilities to **inform trades**:

- Enter **long** positions when a 5-bar pattern historically leads to an upward move.  
- Enter **short** positions when the pattern predicts a decline.

> Must be tested on a **validation set** to avoid overfitting or lookahead bias.

In [5]:
# Group the data by date
daily_groups = data.groupby(raw_data['Timestamp'].dt.date)

# Convert groups to list of dataframes
daily_dfs = [group for _, group in daily_groups]

print(f"Number of trading days: {len(daily_dfs)}")
print("\nSample of first day data:")
print(daily_dfs[0].head())

# Print date range for each day
print("\nAvailable trading days:")
for df in daily_dfs[:5]:
    date = df['Timestamp'].dt.date.iloc[0]
    print(f"- {date}: {len(df)} rows")

Number of trading days: 1256

Sample of first day data:
                  Timestamp    Open    High     Low   Close  Volume        d1  \
0 2020-10-19 09:32:15+00:00  349.16  349.16  349.16  349.16     170       NaN   
1 2020-10-19 09:34:55+00:00  349.06  349.06  349.06  349.06     500 -0.000625   
2 2020-10-19 09:35:25+00:00  349.13  349.13  349.13  349.13     500  0.002333   
3 2020-10-19 09:35:32+00:00  349.11  349.11  349.09  349.10    2500 -0.004286   
4 2020-10-19 09:35:34+00:00  349.11  349.11  349.11  349.11     300  0.005000   

         d2       pct      dpct    target  
0       NaN       NaN       NaN -0.000286  
1       NaN -0.000286       NaN  0.000201  
2  0.000099  0.000201  0.000487 -0.000086  
3 -0.000946 -0.000086 -0.000286  0.000029  
4  0.004643  0.000029  0.000115  0.000000  

Available trading days:
- 2020-10-19: 4888 rows
- 2020-10-20: 4940 rows
- 2020-10-21: 4857 rows
- 2020-10-22: 4923 rows
- 2020-10-23: 4962 rows


In [6]:
combinations = {}
window_size = 5 # last 5 data points
for day in daily_dfs[int(len(daily_dfs)*0.8):]:  # Use last 20% of days for testing
    if len(day) < window_size:
        continue  # Skip days with insufficient data

    for i in range(1+window_size, len(day) - window_size + 1):
        input_seq = day['pct'].iloc[i-window_size:i].values
        combo = ["R" if x < 0 else "G" for x in input_seq] 
        if "".join(combo) not in combinations:
            combinations["".join(combo)] = []
        combinations["".join(combo)].append(day['target'].iloc[i])

In [7]:
probabilities = {}
for combo, targets in combinations.items():
    probabilities[combo] = {"Mean": np.mean(targets), 
                            "Std": np.std(targets), 
                            "Count": len(targets),
                            "Prob_Up": np.sum(np.array(targets) > 0) / len(targets),
                            "Prob_Down": np.sum(np.array(targets) <= 0) / len(targets)}

probabilities = pd.DataFrame(probabilities).T
probabilities = probabilities.sort_index()
print("\nProbabilities for each combination:")
print(probabilities)


Probabilities for each combination:
               Mean       Std    Count   Prob_Up  Prob_Down
GGGGG -2.216572e-07  0.000140  62853.0  0.433965   0.566035
GGGGR -1.202716e-06  0.000150  50487.0  0.436508   0.563492
GGGRG  1.743217e-06  0.000202  52789.0  0.439467   0.560533
GGGRR  8.462533e-07  0.000124  39371.0  0.445531   0.554469
GGRGG -7.778911e-07  0.000208  52934.0  0.437602   0.562398
GGRGR -7.140408e-07  0.000198  43278.0  0.446994   0.553006
GGRRG -2.139012e-06  0.000167  41485.0  0.440858   0.559142
GGRRR  1.958667e-06  0.000233  30923.0  0.447046   0.552954
GRGGG -5.509889e-07  0.000129  52717.0  0.439612   0.560388
GRGGR  3.851658e-07  0.000164  43669.0  0.446106   0.553894
GRGRG -1.553921e-07  0.000225  45422.0  0.441130   0.558870
GRGRR  3.485654e-07  0.000182  33583.0  0.451895   0.548105
GRRGG  5.914788e-07  0.000181  41475.0  0.438095   0.561905
GRRGR -1.058079e-06  0.000199  34220.0  0.450526   0.549474
GRRRG -1.584757e-07  0.000151  32267.0  0.447113   0.552887
GRR

In [8]:
# --- Generate trading signals from `probabilities` (5-bar patterns) and run analyzer ---
# params
window_size = 5
min_count = 20           # minimum historical observations required for a combo
prob_threshold = 0.55    # require Prob_Up >= this to go long, <= 1-this to go short
lookahead_bars = 1       # use next bar's close as fill price (honest execution)


# make working copy and ensure timestamp types are consistent (naive)
df = data.copy().reset_index(drop=True)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
if df['Timestamp'].dt.tz is not None:
    df['Timestamp'] = df['Timestamp'].dt.tz_localize(None)

# prepare next-bar fill price (honest execution)
df['NextBarPrice'] = df['Close'].shift(-lookahead_bars)

def emit(ts, price, sig, new_pos):
    signals.append({
        'Timestamp': ts,
        'Price': price,
        'Signal': sig,
        'Position': new_pos
    })

In [9]:
# Vectorized signal generator (replace slow for-loop with this)
from numpy.lib.stride_tricks import sliding_window_view
import time

def vectorized_signals_from_probabilities(df, probabilities,
                                         window_size=5, min_count=20,
                                         prob_threshold=0.55, lookahead_bars=1):
    t0 = time.time()
    df = df.copy().reset_index(drop=True)
    n = len(df)
    if n <= window_size + lookahead_bars:
        return pd.DataFrame(columns=['Timestamp','Price','Signal','Position'])

    # sign array: 1 for G (>=0), 0 for R (<0)
    s = (df['pct'].values >= 0).astype(np.int8)

    # build sliding windows of length `window_size`; drop the last window which would end at index n
    windows = sliding_window_view(s, window_size)[:-1]  # shape ~ (n-window_size, window_size)
    decision_idx = np.arange(window_size, n)[:windows.shape[0]]  # decision indices matching windows

    # compute integer code for each window (binary with MSB = earliest element)
    weights = (2 ** np.arange(window_size-1, -1, -1)).astype(np.int32)
    codes = windows.dot(weights)  # shape (m,)

    # map probability table index (strings like 'GGRRG') -> code
    def combo_to_code(combo: str):
        bits = [1 if ch == 'G' else 0 for ch in combo]
        return int(''.join(str(b) for b in bits), 2)

    # build mapping dicts from code -> (prob_up, count)
    prob_map = {}
    count_map = {}
    for combo, row in probabilities.iterrows():
        c = combo_to_code(combo)
        prob_map[c] = float(row['Prob_Up'])
        count_map[c] = int(row['Count'])

    # vectorized lookup: for codes not present, fill with nan / 0
    prob_up_arr = np.array([prob_map.get(c, np.nan) for c in codes])
    counts_arr = np.array([count_map.get(c, 0) for c in codes])

    # enforce min_count: invalidate probabilities where count < min_count
    prob_up_arr[counts_arr < min_count] = np.nan

    # desired position at each decision index: 1 long, -1 short, 0 flat
    desired = np.zeros(n, dtype=np.int8)
    # assign only at decision indices
    desired_vals = np.zeros_like(prob_up_arr, dtype=np.int8)
    desired_vals[np.where(prob_up_arr >= prob_threshold)] = 1
    desired_vals[np.where(prob_up_arr <= (1.0 - prob_threshold))] = -1
    desired[decision_idx[:len(desired_vals)]] = desired_vals

    # NextBarPrice aligned
    fill_prices = df['NextBarPrice'].values
    # ensure decision indices have valid fill price
    valid_fill = ~pd.isna(pd.Series(fill_prices[decision_idx]))  # pd.Series used for isna semantics
    # mask out decisions with invalid fills
    desired_vals = desired[decision_idx].copy()
    desired_vals[~valid_fill] = 0
    desired[decision_idx] = desired_vals

    # detect transitions
    prev = np.concatenate(([0], desired[:-1]))
    enters_long_idx = np.where((desired == 1) & (prev != 1))[0]
    enters_short_idx = np.where((desired == -1) & (prev != -1))[0]
    closes_idx = np.where((desired == 0) & (prev != 0))[0]

    # keep only those indices that are valid decision points (>= window_size and < n - lookahead_bars)
    valid_range_mask = (enters_long_idx >= window_size) & (enters_long_idx < n - lookahead_bars)
    enters_long_idx = enters_long_idx[valid_range_mask]
    valid_range_mask = (enters_short_idx >= window_size) & (enters_short_idx < n - lookahead_bars)
    enters_short_idx = enters_short_idx[valid_range_mask]
    valid_range_mask = (closes_idx >= window_size) & (closes_idx < n - lookahead_bars)
    closes_idx = closes_idx[valid_range_mask]

    # Build signals list using vectorized selections
    signals = []
    def append_signals(idxs, sig_name, pos_name):
        for idx in idxs:
            price = fill_prices[idx]
            if pd.isna(price):
                continue
            signals.append({
                'Timestamp': df['Timestamp'].iat[idx],
                'Price': price,
                'Signal': sig_name,
                'Position': pos_name
            })

    append_signals(enters_long_idx, 'LONG', 'LONG')
    append_signals(enters_short_idx, 'SHORT', 'SHORT')
    append_signals(closes_idx, 'CLOSE', 'FLAT')

    t1 = time.time()
    print(f"Vectorized signal generation took {t1 - t0:.3f}s, generated {len(signals)} signals")
    return pd.DataFrame(signals, columns=['Timestamp','Price','Signal','Position'])


# Usage (replace your loop with this call)
trading_signals = vectorized_signals_from_probabilities(df[int(len(df)*0.8):], probabilities,
                                                       window_size=window_size,
                                                       min_count=min_count,
                                                       prob_threshold=prob_threshold,
                                                       lookahead_bars=lookahead_bars)


Vectorized signal generation took 2.790s, generated 337761 signals


In [10]:
print(f"Generated {len(trading_signals)} signals (window={window_size}, min_count={min_count}, thresh={prob_threshold})")
trading_signals.head()

Generated 337761 signals (window=5, min_count=20, thresh=0.55)


,Timestamp,Price,Signal,Position
0,2024-10-14 14:18:00,582.833600,SHORT,SHORT
1,2024-10-14 14:18:19,582.910000,SHORT,SHORT
2,2024-10-14 14:18:34,582.820000,SHORT,SHORT
3,2024-10-14 14:18:43,582.840000,SHORT,SHORT
4,2024-10-14 14:18:45,582.849292,SHORT,SHORT


In [11]:
# Normalize timestamps to timezone-naive (run this before any comparisons/joins)
def ensure_naive_ts(df, col='Timestamp'):
    df = df.copy()
    df[col] = pd.to_datetime(df[col])
    # If series is tz-aware, drop the timezone
    try:
        if df[col].dt.tz is not None:
            df[col] = df[col].dt.tz_localize(None)
    except Exception:
        # fallback: try tz_convert(None) for some pandas versions
        try:
            df[col] = df[col].dt.tz_convert(None)
        except Exception:
            pass
    return df

raw_data = ensure_naive_ts(raw_data, 'Timestamp')
data = ensure_naive_ts(data, 'Timestamp')

# Quick sanity check
print("raw_data Timestamp dtype:", raw_data['Timestamp'].dtype)
print("data Timestamp dtype:", data['Timestamp'].dtype)

raw_data Timestamp dtype: datetime64[ns]
data Timestamp dtype: datetime64[ns]


# Portfolio Analysis

In [12]:
from trading_analyzer import TradingAnalyzer
analyzer = TradingAnalyzer(trading_signals, raw_data)


In [13]:
analyzer.get_statistics()

{'basic': {'Total Trades': 168881,
  'Winning Trades': 82542,
  'Losing Trades': 74590,
  'Win Rate': 0.4887583564758617,
  'Avg Return': 6.149420458551348e-07,
  'Avg Win': 0.00011680197604341378,
  'Avg Loss': -0.0001278618672600804,
  'Max Drawdown': -0.1410183814893871,
  'Final Equity': 109280.28261587046,
  'Total Return': 0.09280282615870461,
  'Sharpe': 0.023012140267392095},
 'advanced': {'Annualized Volatility': 0.006734071396873847,
  'Skewness': 22.429941946106943,
  'Kurtosis': 5703.168255931675,
  'VaR(95)': -0.0002482544608222935,
  'CVaR(95)': -0.0006013340707926065,
  'Best Trade': 0.06950042984048152,
  'Worst Trade': -0.039515653887568405,
  'Monthly Returns': YearMonth
  2024-10    0.039186
  2024-11   -0.012976
  2024-12   -0.006883
  2025-01   -0.028013
  2025-02    0.002520
  2025-03    0.075606
  2025-04    0.125501
  2025-05    0.023220
  2025-06   -0.068216
  2025-07   -0.012732
  2025-08   -0.026712
  2025-09    0.019163
  2025-10   -0.025811
  Name: Return, 

analyzer.plot_price_with_slider(window_minutes=1)

In [ ]:
figures = analyzer.plot_performance(use_plotly=True)
for title, fig in figures.items():
    print(f"\n{title}")
    fig.show()

In [ ]:
trades = analyzer.get_best_worst(n=5)
print("\nBest 5 Trades:")
print(trades['best'])
print("\nWorst 5 Trades:")
print(trades['worst'])